# Naples, 1980s — mapping musical memory extracted from language models

This notebook takes a 20-item oral-history questionnaire about music places in 1980s Naples
(record shops, clubs, meeting places with a jukebox, live venues, concert memories), fills it with
**synthetic respondents generated by two language models**, and turns the answers into a map.

**What this measures.** Not what Neapolitans remember. It measures what the models "remember" about
the city: which places they can name, in which quarter they put them, and how confident that memory is.
Every row therefore carries `modello`, `modello_id`, `prompt_versione`, `sintetico = SI` and a
`confidenza_luoghi` flag. Treat the result as a map of *model memory*, and read it next to real
interviews, never instead of them.

**Pipeline.** sheet → pandas → optional LLM normalisation of place names → zone centroids / geocoding
→ interactive map + charts → export.

**Hosted interactive map:** <https://bperak.github.io/napoli-1980-map/> (same file the map cell writes).


## 0. Setup

Everything below is defensive: a cell prints what it could not do instead of stopping the notebook.


In [ ]:
!pip install -q --upgrade folium gspread pandas matplotlib requests 2>/dev/null
print("dependencies ready")

In [ ]:
import io, json, re, time, unicodedata, warnings
from collections import Counter
import pandas as pd
import requests
warnings.filterwarnings("ignore")

# ---- configuration -------------------------------------------------------
SHEET_CSV_URL = ("https://docs.google.com/spreadsheets/d/1BRatPazDttqVVxpFfEBxuOd4ddiA9_mDleRNDJQtQYU/export?format=csv")
LOCAL_FALLBACK = "napoli1980_synthetic.csv"   # used only if the sheet is unreachable
MODEL_COL = "modello"
GOOGLE_API_KEY = ""
try:
    from google.colab import userdata           # noqa
    GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    print("Gemini key found in Colab Secrets")
except Exception:
    print("No Colab Secrets — LLM normalisation will be skipped, heuristics will run instead")
# Colab ima display(); obični Python nema — osiguraj da ništa ne padne
try:
    display
except NameError:
    def display(obj):
        try:
            print(obj.to_string() if hasattr(obj, "to_string") else str(obj)[:2000])
        except Exception:
            print(str(obj)[:2000])

print("config ready")

### 0b. Offline copy of the dataset

This cell carries the whole dataset inside the notebook (110 kB compressed) and writes it next to the notebook. It exists for one reason: a live demonstration should not depend on the Wi-Fi of the room. If the sheet is reachable, the next cell uses the sheet; if it is not, it falls back to this file.


In [ ]:
# offline copy — only used when the sheet cannot be reached
import base64 as _b64, gzip as _gz, pathlib as _pl
OFFLINE_CSV_GZ = "..."
_p = _pl.Path("napoli1980_synthetic.csv")
if not _p.exists():
    _p.write_bytes(_gz.decompress(_b64.b64decode(OFFLINE_CSV_GZ)))
print(f"offline copy ready: {_p.stat().st_size // 1024} KB")


## 1. Load the data

Two ways in: the published CSV of the sheet, or a local copy uploaded next to the notebook.
If neither works the notebook generates a tiny demo frame so the rest still runs.


In [ ]:
def load_frame():
    """Sheet -> CSV export with a browser UA (Google serves an interstitial to unknown clients),
    then a local copy, then a demo row. Never raises."""
    ua = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"}
    try:
        r = requests.get(SHEET_CSV_URL, headers=ua, timeout=45)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        if "modello" not in df.columns:
            raise ValueError("CSV did not contain the expected columns")
        print(f"loaded from sheet: {df.shape[0]} rows x {df.shape[1]} columns")
        return df, "sheet"
    except Exception as e:
        print("sheet unreachable:", str(e)[:140])
    try:
        df = pd.read_csv(LOCAL_FALLBACK)
        print(f"loaded from local file: {df.shape[0]} rows")
        return df, "local"
    except Exception as e:
        print("local file unavailable:", str(e)[:120])
    demo = pd.DataFrame({
        "respondent_id": ["demo-001-01"],
        "anno_nascita": [1962], "genere": ["Maschio"],
        "zona_residenza": ["Vomero"], "negozio_nome": ["Sudigiri"],
        "negozio_zona": ["Centro storico"], "club_nome": ["Flamingo"],
        "club_zona": ["Fuorigrotta / Bagnoli"], "teleportazione": ["demo row"],
        "confidenza_luoghi": ["media"], "modello": ["demo"]})
    print("running on a demo row only — upload the real sheet to see the city")
    return demo, "demo"

df, SOURCE = load_frame()
display(df.head(3))
print("\nrows per model:")
print(df[MODEL_COL].value_counts().to_string() if MODEL_COL in df.columns else "column missing")

## 2. Audit before mapping

What matters for a map: does each row carry a place name and a quarter, and how confident is the model
about the place it invented or recalled?


In [ ]:
rows = []
rows.append(("rows total", len(df)))
for col in ["zona_residenza", "negozio_nome", "negozio_zona", "club_nome", "club_zona"]:
    if col in df.columns:
        filled = df[col].notna() & (df[col].astype(str).str.len() > 1)
        rows.append((f"{col} filled", f"{filled.mean():.0%}"))
if "confidenza_luoghi" in df.columns:
    rows.append(("confidence: " + ", ".join(f"{k}={v}" for k, v in
                 df["confidenza_luoghi"].value_counts().items()), ""))
audit = pd.DataFrame(rows, columns=["check", "value"])
display(audit)

if "anno_nascita" in df.columns:
    years = pd.to_numeric(df["anno_nascita"], errors="coerce").dropna()
    if len(years):
        print(f"birth years: {int(years.min())}-{int(years.max())}, median {int(years.median())}")
dupes = df.duplicated(subset=[c for c in ["negozio_nome", "teleportazione"] if c in df.columns]).sum()
print(f"exact duplicate place+anecdote pairs: {dupes}")

## 3. Place extraction

The questionnaire stores places as free text (a name, sometimes with the street). For the map we need
`place → type → quarter`. With a Gemini key the normalisation is done by a model; without one, a
heuristic parser splits the string. Both paths write the same columns, so the map never depends on the key.


### 3a. Heuristic path (always available)

In [ ]:
PLACE_TYPES = {
    "negozio": ["negozio", "dischi", "musicassette", "rivendita", "vinile"],
    "club": ["club", "discoteca", "balera", "night"],
    "ritrovo": ["bar", "piazza", "muretto", "piazzetta", "gelateria"],
    "live": ["live", "jam", "concerto", "palazzetto", "stadio", "teatro"],
}


def clean(text):
    if not isinstance(text, str):
        return ""
    t = unicodedata.normalize("NFKD", text)
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", t).strip()


def heuristic_places(frame):
    out = []
    for _, r in frame.iterrows():
        for kind, zone_col in (("negozio", "negozio_zona"), ("club", "club_zona")):
            name = r.get(f"{kind}_nome", "")
            zone = r.get(zone_col, r.get("zona_residenza", ""))
            if isinstance(name, str) and len(name) > 1:
                out.append({"respondent_id": r.get("respondent_id", ""),
                            "modello": r.get(MODEL_COL, ""),
                            "tipo": kind,
                            "luogo": name.strip()[:80],
                            "zona": str(zone)[:80],
                            "confidenza": r.get("confidenza_luoghi", ""),
                            "racconto": str(r.get("teleportazione", ""))[:400],
                            "metodo": "heuristic"})
        for kind, col in (("ritrovo", "ritrovo_bar"), ("live", "locale_live")):
            txt = r.get(col, "")
            if isinstance(txt, str) and len(txt) > 3:
                out.append({"respondent_id": r.get("respondent_id", ""),
                            "modello": r.get(MODEL_COL, ""),
                            "tipo": kind,
                            "luogo": txt.strip()[:80],
                            "zona": str(r.get("zona_residenza", ""))[:80],
                            "confidenza": r.get("confidenza_luoghi", ""),
                            "racconto": txt[:400],
                            "metodo": "heuristic"})
    return pd.DataFrame(out)

places = heuristic_places(df)
print(f"{len(places)} places extracted by heuristic")
display(places.head(5))

### 3b. Model path (optional, only if a Gemini key exists)

One request per batch of place strings; the model returns a JSON list with a normalised name, a type and
a quarter. Anything the model cannot parse stays as it came from the heuristic pass.


In [ ]:
def gemini_normalise(batch_texts, api_key):
    url = ("https://generativelanguage.googleapis.com/v1beta/models/"
           "gemini-3.8-flash:generateContent")
    instr = ("Normalise these Neapolitan place names from 1980s memories. "
             "Return ONLY a JSON array of objects with keys: original, nome, tipo "
             "(negozio|club|ritrovo|live|altro), zona. No prose.\n" + json.dumps(batch_texts, ensure_ascii=False))
    payload = {"contents": [{"parts": [{"text": instr}]}]}
    r = requests.post(url, json=payload, headers={"x-goog-api-key": api_key}, timeout=60)
    r.raise_for_status()
    text = r.json()["candidates"][0]["content"]["parts"][0]["text"]
    m = re.search(r"\[.*\]", text, re.S)
    return json.loads(m.group(0)) if m else []

if GOOGLE_API_KEY and len(places):
    uniq = places["luogo"].dropna().unique().tolist()[:60]
    try:
        enriched = gemini_normalise(uniq, GOOGLE_API_KEY)
        mapping = {clean(e.get("original", "")): e for e in enriched if isinstance(e, dict)}
        def apply(row):
            e = mapping.get(clean(row["luogo"]))
            if not e:
                return row
            return {**row, "luogo": e.get("nome") or row["luogo"],
                    "tipo": e.get("tipo") or row["tipo"],
                    "zona": e.get("zona") or row["zona"], "metodo": "gemini"}
        places = pd.DataFrame([apply(r) for _, r in places.iterrows()])
        print(f"normalised {len(mapping)} of {len(uniq)} unique places with Gemini")
    except Exception as e:
        print("Gemini normalisation skipped:", str(e)[:150])
else:
    print("no key or no places — heuristic results stand")
display(places["tipo"].value_counts().to_string())

## 4. From quarter to coordinates

Quarters are mapped to approximate centroids (hand-made table below, good enough for a city-scale map).
Specific places get a real geocode from OpenStreetMap when the name resolves; otherwise the quarter
centroid is used, and the row is marked `geo = quarter` so the map does not pretend to a precision it
does not have.


In [ ]:
ZONE_COORDS = {
    "centro storico": (40.8518, 14.2580), "decumani": (40.8518, 14.2580),
    "piazza bellini": (40.8518, 14.2580), "carlo iii": (40.8600, 14.2730),
    "area orientale": (40.8600, 14.2730), "vomero": (40.8480, 14.2300),
    "arenella": (40.8570, 14.2230), "rione alto": (40.8570, 14.2230),
    "fuorigrotta": (40.8270, 14.1900), "bagnoli": (40.8250, 14.1750),
    "chiaia": (40.8330, 14.2280), "mergellina": (40.8330, 14.2280),
    "posillipo": (40.8100, 14.2100), "sanita": (40.8580, 14.2500),
    "vergini": (40.8580, 14.2500), "poggioreale": (40.8650, 14.2900),
    "barra": (40.8700, 14.3000), "secondigliano": (40.8900, 14.2700),
    "scampia": (40.8950, 14.2600), "soccavo": (40.8500, 14.1900),
    "pianura": (40.8550, 14.1700), "porto": (40.8450, 14.2650),
    "pendino": (40.8450, 14.2650), "materdei": (40.8560, 14.2400),
}

def zone_lookup(zone_text):
    t = clean(str(zone_text)).lower()
    for key, coords in ZONE_COORDS.items():
        if key in t:
            return coords
    return None

def jitter(coords, seed):
    if not coords:
        return None
    rnd = random.Random(seed)
    return (coords[0] + rnd.uniform(-0.004, 0.004), coords[1] + rnd.uniform(-0.004, 0.004))

import random  # noqa: E402  (used by jitter)
places["coords"] = places["zona"].apply(zone_lookup)
places["lat"] = [jitter(c, i)[0] if jitter(c, i) else None for i, c in enumerate(places["coords"])]
places["lon"] = [jitter(c, i)[1] if jitter(c, i) else None for i, c in enumerate(places["coords"])]
places["geo"] = ["quarter" if c else "unknown" for c in places["coords"]]
print(f"mapped {places['lat'].notna().sum()} of {len(places)} places to a quarter centroid")
print(f"unmapped rows: {places['lat'].isna().sum()} (they stay out of the map, not out of the tables)")

## 5. The map

Marker clusters per place, coloured by model, popup with the anecdote and the confidence flag.
A heat layer on top shows where the two models concentrate their memory.


In [ ]:
import folium
from folium.plugins import MarkerCluster, HeatMap

MODEL_COLORS = {}
PALETTE = ["#1f3864", "#b03a2e", "#186a3b", "#7d3c98", "#b9770e"]
for i, m in enumerate(sorted(places["modello"].dropna().unique())):
    MODEL_COLORS[m] = PALETTE[i % len(PALETTE)]

# Carto podloge traže API ključ (bez njega vrati prazno), zato OSM + Esri sivi sloj
m = folium.Map(location=[40.8518, 14.2580], zoom_start=12, tiles="OpenStreetMap",
               attr="© OpenStreetMap contributors")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/"
          "MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors", name="Esri sivi", show=False,
).add_to(m)
valid = places.dropna(subset=["lat", "lon"])

clusters = {t: MarkerCluster(name=t).add_to(m) for t in valid["tipo"].dropna().unique()}
for _, r in valid.iterrows():
    popup = folium.Popup(
        f"<b>{r['luogo']}</b><br>{r['tipo']} · {r['zona']}<br>"
        f"<i>{str(r['racconto'])[:220]}</i><br>"
        f"<small>model: {r['modello']} · confidence: {r['confidenza']} · geo: {r['geo']}</small>",
        max_width=320)
    folium.CircleMarker(
        location=[r["lat"], r["lon"]], radius=6, popup=popup,
        color=MODEL_COLORS.get(r["modello"], "#555"), fill=True, fill_opacity=0.75,
        weight=1).add_to(clusters.get(r["tipo"], m))

if len(valid) > 2:
    HeatMap(valid[["lat", "lon"]].values.tolist(), radius=18, blur=14).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save("napoli1980_map.html")
print(f"map written: napoli1980_map.html ({len(valid)} points)")

# prikaz karte unutar ćelije: HTML iz foliuma, s rezervom ako IPython nije dostupan
try:
    from IPython.display import HTML
    display(HTML(m._repr_html_()))
except Exception as e:
    print("inline HTML nije moguć, koristi se objekt karte:", str(e)[:80])
    display(m)

### The same map, hosted — interactive inside this notebook

The live copy of the map is served at <https://bperak.github.io/napoli-1980-map/>. It is rendered below as a frame, so it can be zoomed, clicked and filtered without leaving the notebook. The cell above builds the same map from the data and writes it to `napoli1980_map.html`; this one simply shows the published version.


In [ ]:
HOSTED_MAP_URL = "https://bperak.github.io/napoli-1980-map/"
try:
    from IPython.display import HTML, display
    display(HTML(f'<div style="font:12px sans-serif;color:#555;margin:4px 0 6px">'
                  f'live map: <a href="{HOSTED_MAP_URL}" target="_blank">{HOSTED_MAP_URL}</a></div>'
                  f'<iframe src="{HOSTED_MAP_URL}" width="100%" height="560" '
                  f'style="border:1px solid #ccd;border-radius:6px" loading="lazy"></iframe>'))
except Exception:
    print("Apri la mappa pubblicata nel browser:", HOSTED_MAP_URL)


## 6. What the map says

Two views that matter more than the dots: which quarters dominate each model's memory, and how much the
two models agree with each other.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
zone_counts = (places.assign(zona=places["zona"].str.lower().str[:28])
               .groupby(["zona", "modello"]).size().unstack(fill_value=0))
zone_counts.sum(axis=1).sort_values(ascending=False).head(10).plot(
    kind="barh", ax=axes[0], color="#1f3864")
axes[0].set_title("Places per quarter (all models)")
axes[0].invert_yaxis(); axes[0].set_xlabel("places")

places["tipo"].value_counts().plot(kind="bar", ax=axes[1], color="#b03a2e")
axes[1].set_title("Places per type"); axes[1].set_xlabel("")
plt.tight_layout(); plt.show()

print("\nper model:")
print(places.groupby("modello").agg(places=("luogo", "count"),
                                    quarters=("zona", "nunique"),
                                    low_conf=("confidenza", lambda s: (s == "bassa").mean())).to_string())

In [ ]:
# overlap between models: how much of the city do they "remember" the same way?
def place_set(model):
    return {clean(x).lower() for x in places.loc[places["modello"] == model, "luogo"].dropna()}

models = sorted(places["modello"].dropna().unique())
if len(models) >= 2:
    a, b = place_set(models[0]), place_set(models[1])
    inter, union = a & b, a | b
    print(f"{models[0]}: {len(a)} distinct places")
    print(f"{models[1]}: {len(b)} distinct places")
    print(f"shared: {len(inter)}  |  Jaccard overlap: {len(inter)/max(1,len(union)):.0%}")
    if inter:
        print("shared examples:", ", ".join(sorted(inter)[:12]))
else:
    print("only one model in the data — overlap not computable")

## 7. Export

The enriched table (places + coordinates + method) is what the map is built from; keep it next to the
notebook so the figure can be reproduced without re-running the extraction.


In [ ]:
out_csv = "napoli1980_places_enriched.csv"
places.to_csv(out_csv, index=False, encoding="utf-8")
print(f"written {out_csv}: {len(places)} rows")
try:
    from google.colab import files
    files.download(out_csv)
except Exception:
    print("not running in Colab — file stays in the working directory")
display(places.head(3))

## 8. The sources, and what each one can actually give

The map above is one artefact of a larger question: *where were the music places of 1980s Naples?*
No model answers that. Sources do. The table below is the working inventory — every channel that was
checked, what it yields, and what it costs in time or access. The last column is the honest one: it says
whether the channel can produce an **address** (level A), a **date and a name** (level B), a **testimony**
(level C), or nothing.

The ordering is by return on effort, not by prestige. A directory beats a newspaper for the first step,
because a directory gives name, street and telephone without reading a single sentence of prose.


In [ ]:
# SOURCES — inventario dei canali (aggiornato al 21 ottobre 2026)
import pandas as pd

SOURCES = [
    ("Pagine Gialle / elenco telefonico 1980–1989", "volumi cartacei: Biblioteca Nazionale di Napoli, antiquariato",
     "nome + via + telefono per categoria (sale da ballo, negozi di dischi)", "A", "alto: serve presenza fisica"),
    ("Comune di Napoli — licenze pubblico spettacolo / SUAP", "archivio comunale, richiesta scritta",
     "titolo, indirizzo, titolare, anno", "A", "medio: richiesta formale"),
    ("SIAE — archivio permessi spettacolo", "ufficio SIAE competente",
     "data, luogo, organizzatore di concerti e serate", "A/B", "medio: richiesta formale"),
    ("Camera di Commercio — visure storiche", "Registro imprese, visura storica a pagamento",
     "sede legale di società con nome del locale", "A", "basso costo, copertura irregolare"),
    ("Il Mattino 1978–1989", "Emeroteca Tucci (in sede) / archivio digitale in abbonamento",
     "inserzioni di concerti, aperture, pubblicità di negozi: il record continuo", "B", "alto: abbonamento o sede"),
    ("Stampa musicale d'epoca (Il Mucchio, Rockerilla, Ciao 2001)", "emeroteche, collezioni private, Internet Archive",
     "rubriche di scena locale, recensioni, indirizzi", "B", "medio"),
    ("Fanzine locali (es. Megawave)", "archivi privati, Internet Archive, collezionisti",
     "nomi di locali, date di serate, foto", "B/C", "alto: reperimento difficile"),
    ("Archivio fotografico Carbone", "archiviocarbone.it, richiesta di consultazione",
     "foto datate con luogo: insegne, incroci, interni", "B", "medio: richiesta"),
    ("Radio (Radio Kiss Kiss, radio libere)", "archivio dell'emittente, registrazioni private",
     "serate trasmesse in diretta da un club: nome e data", "B/C", "alto: archivi non ordinati"),
    ("Guide e letteratura (Ten Cities, guide cittadine)", "biblioteche, acquisto",
     "nomi, vie, anni, racconti di protagonisti", "B", "basso"),
    ("Discogs / MusicBrainz", "API pubbliche, crediti di etichetta e note di copertina",
     "studi, etichette, indirizzi citati nei crediti", "B/C", "basso"),
    ("Testimonianze orali strutturate", "raccolta con consenso, due testimoni indipendenti",
     "nome, zona, anno, aneddoto databile", "C", "medio: organizzazione"),
    ("Gruppi e forum di memoria urbana", "Facebook, forum locali",
     "foto, volantini, ricordi datati", "C/D", "basso, ma richiede etica e verifica"),
    ("YouTube e archivi video amatoriali", "ricerca per locale e anno, commenti",
     "filmati di serate, didascalie, commenti con nomi e date", "C", "basso"),
    ("Modelli linguistici (DeepSeek, Gemma, Gemini)", "questo progetto",
     "ipotesi da verificare: mai una fonte", "D", "quasi zero"),

]
src = pd.DataFrame(SOURCES, columns=["canale", "dove", "cosa dà", "livello", "costo"])
display(src)
print(src["livello"].value_counts().to_string())

### What the sources have produced so far

Sixteen places are attested in published sources, ten of them with a street or at least a quarter, so ten
can be put on a map. The rest stay in the table: placing them would be an invention of the same kind the
project is trying to avoid. Levels: **B** = two independent published sources; **C** = one source, or a
source that names the place without an address; **A** = a document with an address produced at the time
(none yet — that is what the directories are for).


In [ ]:
# ATTESTATI — luoghi nominati dalle fonti (estratto; il foglio completo è nel registro)
ATTESTATI = [["Diamond Dogs", "club underground", "Cavone, ipogeo (cisterne acquedotto)", "1984–1987/88", "B", "D. Capasso, in Ten Cities (Goethe-Institut / Spector Books), estratto su Rolling Stone; la"], ["ZX", "punk club", "via Atri", "≈1979–1983", "B", "l'Indiependente, 'Spacca Napoli #3'; Corriere Napoli (cultura)"], ["Trilogy", "club", "collina del Vomero", "inizi anni '80", "C", "D. Capasso, Rolling Stone (estratto Ten Cities)"], ["Pulsar", "club", "via Santa Maria di Costantinopoli", "inizi anni '80", "C", "D. Capasso, Rolling Stone (estratto Ten Cities)"], ["Caffè della Luna", "bar/club", "Chiaia", "inizi anni '80", "C", "D. Capasso, Rolling Stone (estratto Ten Cities)"], ["Kiss Kiss", "discoteca", "Napoli (obitelj Niespolo)", "od 1972", "B", "D. Capasso, Rolling Stone; voci.fm (povijest Radio Kiss Kiss)"], ["Galapagos", "discoteca", "Vomero / Chiaia / Posillipo (bez adrese)", "anni '80", "C", "D. Capasso, Rolling Stone"], ["Papillon", "discoteca", "Vomero / Chiaia / Posillipo (bez adrese)", "anni '80", "C", "D. Capasso, Rolling Stone"], ["Piccadilly", "discoteca", "Vomero / Chiaia / Posillipo (bez adrese)", "anni '80", "C", "D. Capasso, Rolling Stone"], ["La Mela", "discoteca", "Vomero / Chiaia / Posillipo (bez adrese)", "anni '80", "C", "D. Capasso, Rolling Stone"], ["De Marco Dischi", "negozio dischi", "Centro Storico", "anni '80", "C", "l'Indiependente, 'Spacca Napoli #3'"], ["Disco d'Oro", "negozio dischi", "via Raffaele Morghen 31 (od 1981.), prije via Marconi", "od 1976", "B", "Corriere Viaggi, pregled povijesnih trgovina diskovima u Italiji"], ["Shaker Club", "night club", "via Nazario Sauro", "1950-e (kontekst)", "B", "D. Capasso, Rolling Stone"], ["Lloyd", "night club", "via Partenope", "1950-e (kontekst)", "B", "D. Capasso, Rolling Stone"], ["Teatro Ausonia", "teatro / koncerti", "Napoli (centro)", "studeni 1984.", "C", "D. Capasso, Rolling Stone"], ["Segnali di Accelerazione", "centro sociale", "Acerra (provincija)", "sredina '80-ih", "C", "D. Capasso, Rolling Stone"]]

att = pd.DataFrame(ATTESTATI, columns=["luogo", "tipo", "via / zona", "anni", "livello", "fonte"])
display(att)
print("attestati:", len(att), "| per livello:", att["livello"].value_counts().to_dict())
print("localizzabili (via o quartiere):", int(att["via / zona"].str.contains("via |quartiere|Cavone|Chiaia|Vomero|Sanità|Acerra|Centro", case=False).sum()))

### The extractor: how a page becomes rows

`estrattore.py` (in the Drive folder) takes the text of an archive page and asks a model for places in a
fixed schema — name, type, street, quarter, years — **and for the exact sentence where each place is named**.
The sentence is the evidence; the row can be checked without re-reading the page. Tested on the 1980s
section of Capasso's chapter on Naples clubbing, it returned seven places in about forty seconds:
Trilogy (Vomero), Pulsar (via Costantinopoli), Caffè della luna (Chiaia), Diamond Dogs (11 June 1984),
Segnali di Accelerazione (Acerra), Teatro Ausonia, Kiss Kiss (1972) — every one with its own quote.

The division of labour is the point: the model reads and proposes, the quotation is the proof, the
historian decides. Pointed at OCR'd pages of Pagine Gialle or Il Mattino, the same function returns the
same schema, one row per venue, with the page it came from.

```python
# the prompt, in short (full version in estrattore.py)
# "Estrai SOLO i luoghi musicali realmente nominati nel testo ... per ognuno: nome, tipo, via,
#  quartiere, anni, e la frase esatta (max 25 parole) in cui il luogo è nominato."
```


## 9. How to read this honestly

- **The respondents are synthetic.** No person was interviewed. The rows say what DeepSeek and Gemma 4
  produce when asked to remember 1980s Naples, with model, prompt version and date stored per row.
- **Confidence is self-reported by the model.** `confidenza_luoghi = bassa` marks places the model itself
  flagged as plausible rather than documented; those dots deserve a different reading than the others.
- **The geography is coarse.** Points sit on quarter centroids with a small jitter, so a dense cluster
  means "many memories placed in this quarter", not "this exact corner".
- **What the comparison is good for.** Differences between the two models (which quarters, which venues,
  how much they overlap) are a readable signal about training data, not about the city.
- **What it must be paired with.** Real interviews, the ARCHAEODIGIT archive, or a documented list of
  closed venues, before any claim about Neapolitan musical memory is made.


## 10. The extractor, running on real documents

Everything above is generated. This part is not: eight real documents are the base — Capasso's chapter on Neapolitan clubbing, the punk chapter of *Spacca Napoli #3*, the Madre and Frizzifrizzi pieces on the Diamond Dogs, Capobianco's memoir, the Radio Kiss Kiss oral-history transcript, the Wikipedia entry and the TTC Club history — and the extractor reads them and returns a table of places, each with **the sentence it came from**. The docs are stored in the notebook so the result can be checked without a network.

The extractor needs a model. It will use the first key it finds in **Colab Secrets**: `GEMINI_API_KEY` (Google AI Studio), `DEEPSEEK_API_KEY`, or the two Cloudflare Access values for the local Gemma on the DGX Spark (`GEMMA4_CF_CLIENT_ID`, `GEMMA4_CF_CLIENT_SECRET`). Without a key the cell says so instead of inventing anything — the saved output below is what the same code returned when it was run.


In [ ]:
DOCS = {
 "Capasso, Ten Cities — estratto Rolling Stone (1980s)": "l’identità sonora. In questo modo, ad esempio, nascono le correnti culturali che cercano di riagganciare la città ad un flusso cosmopolita, non identitario, di allontanarsi dall’influenza forte della politica nella cultura, per trovare nella musica spazi di evoluzione creativa, di incontro al di là delle ideologie e degli steccati sociali. I segnali di questa necessità sono favoriti e accompagnati dalla cultura punk, da una tendenza che spesso porterà Napoli a intrattenere contatti diretti con le città di riferimento di questo movimento, in particolare con Londra e il suo carico di nuova musica, per importare artisti, dischi, mode. Di questi tempi in città si ascolta il pop italiano e la disco music nelle discoteche, esistono ancora i puristi del rock progressivo degli anni ’70, gli orfani della psichedelica post hippie, e il jazz, che in città è da sempre molto seguito e suonato. Nel programma radiofonico di Salvio e Salvatore, chiamato Speed of Life, ci sono i primi tentativi di proporre attraverso la radio suoni diversi, programmando musiche di Wall of Voodoo, Echo & Bunnymen, The Clash, Psychedelic Furs, Iggy Pop. Questo sempre all’inizio degli anni ’80, quando in città si vedevano i primi locali punk come lo ZX di via Atri in pieno Centro Storico, il Trilogy sulla collina del Vomero (dove troviamo in consolle Gigi D’Aria, il primo dj alternativo della scena), il Pulsar di via Costantinopoli, il Caffè della luna nel quartiere di Chiaia. Nello stesso tempo, Salvatore Magnoni assieme a Salvio Cusano ed altri amici, cercano un club per proporre i nuovi generi di musicali aperti alla ricerca e, con una buona dose di spontaneità organizzativa, l’11 giugno 1984 aprono il Diamond Dogs. Sono ragazzi di diversa estrazione sociale provenienti da diversi quartieri della città come il Vomero, Piscinola, Soccavo, e da alcune aree della provincia, Racconta Salvatore: “C’erano discussioni fra noi. C’era l’ala più radicale dalle scelte estreme, e c’era la voglia di pop, così come ci furono i mercoledì di Nicola Catalano, Atrocity Exhibition, rumori industriali per pochi, e i sabati con la dance di Patrizio Squeglia. Fra noi c’era il tardo sessantotto e i postumi del settantasette. C’era la classe operaia, la piccola borghesia, l’artigiano di provincia, la middle class del Vomero e il jet set di Chiaia. I riferimenti culturali tracciavano una mappa ampia, nella geografia mutevole della sinistra dei primi anni ottanta, tirata da una parte dai puristi delle feste dell’unità, dei cantautori e del progressive, dall’altra dal politicamente scorretto della nuova onda, dove riviste come Il Male e Frigidaire mischiavano le carte. Nel Diamond Dogs coesistevano queste spinte, gli occhialini tondi di Luca e gli anfibi neri di Tonino “Piccolone”, i capelli lunghi e biondi di Stefania e quelli corti, squadrati, di Sandra.” Il Diamod Dogs è il classico genere di locale che possiamo trovare nelle aree della città storica anche oggi, ricavato in spazi ipogei in genere formati da sequenze di piccole stanze, cantine e sotterranei di palazzi antichi, o nel caso del Diamond Dogs stesso, da un utilizzo creativo delle cisterne dell’antico acquedotto greco/romano in disuso e cave di tufo, che caratterizzano il poroso sottosuolo della città. Sono spazi nascosti, di rado visibili dalla strada, spesso celati all’interno di cortili, dove passando da piccole porte e scendendo rampe di scale si poteva accedere in questi mondi nascosti. Ricorda ancora Salvatore: “la postazione del dj era inizialmente sul mezzanino, poi l’anno dopo sotto, dietro un frontale di furgone. Tutto il locale è scavato nel tufo. Sotto la postazione del dj abbiamo messo il bar, che poi sarebbe un bancone frigo che non riesce a raffreddare le bevande, che sono poche e finiscono subito, e qualche sacco di carta per alimenti con centinaia di panini da riempire. Non c’è attacco di acqua, corrente, nessun cocktail, nessun bicchiere di vetro. Lattine e bottiglie di birra, una micidiale affettatrice a mano per formaggi e prosciutti. Igiene da nave pirata. Le pareti non sono intonacate. Le abbiamo coperte di calce che costava di meno, ma ben presto l’alto tasso di umidità presente nelle grotte miscelato con il calore dei corpi fa liquefare la calce che si sposta sugli abiti di chiunque sfiori le pareti. Pareti sulle quali ognuno di noi ha disegnato delle cose, corpi danzanti, stelle cadenti. Sono affreschi primitivi, raccontano i nostri desideri, la voglia di incontri. Anche questi disegni dureranno poco, il tempo di scivolare via, come sogni al mattino”. Salvatore ricorda una certa difficoltà della sinistra di leggere il contemporaneo, di riuscire a codificare stili e generi nel loro reale contesto culturale. Questo ha portato una parte dei giovani di sinistra, sia negli anni ‘80 che dopo, nei primi ‘90, a considerare con sospetto e distanza le rivoluzioni del punk, della new wave e più tardi nei confronti della musica elettronica e della techno. Non solo per diversità estetiche, ma di posizionamento politico e di modalità di aggregazione. La musica per la sinistra doveva essere musica di denuncia diretta, di impegno sociale esplicito, non poteva parlare d’amore, non poteva essere semplicemente strumentale, disimpegnata, troppo edonista, o peggio nichilista. Questo limite culturale è uno dei principali motivi che spinge la nascita di spazi come il Diamond Dogs, uno dei locali underground di Napoli che ancora riverbera storie e ricordi, di sicuro un punto di svolta nella storia del Clubbing partenopeo, capace di condensare le molteplici tensioni di superamento di tali limiti culturali socialmente sedimentati. Il Diamond Dogs è quindi uno spazio inquietante specialmente dal punto di vista politico della sinistra, si tratta di uno spazio topograficamente anomalo, non codificabile, un’eterotopia che mette in discussione i riferimenti tradizionali del conflitto sociale, quantomeno nel contesto italiano e nello specifico napoletano. Il Diamond Dogs dura fino al 1988, e nel frattempo crea anche un gemellaggio con uno dei primi centri sociali aperti all’epoca, Segnali di Accellerazione, localizzato in provincia nella città di Acerra. Questo gemellaggio porterà uno scambio molto proficuo con la periferia, un flusso di stimoli e relazioni che segnò la strada anche per futuri centri sociali cittadini e nuovi locali periferici. Purtroppo in questo magma di creatività e confusione fa la sua comparsa l’eroina, la droga che minerà dall’interno la spinta creativa e le possibili evoluzioni di quanto stava accadendo e che prenderà in questi anni il sopravvento per diventare un problema sociale molto serio. All’inizio degli anni ottanta l’eroina era molto più diffusa nella provincia, la criminalità aveva centri di distribuzione specialmente nell’area vesuviana, Ercolano, oltre ad essere la sede del più importante mercatino di vestiti usati della regione, dove generazioni di alternativi hanno inventato i loro guardaroba, era, ed è stata per lungo tempo anche un importante piazza di spaccio. In questi anni, prima del dilagare dell’eroina, le droghe più diffuse erano hashish, l’erba, le anfetamine e gli acidi. L’ecstasy sarebbe apparsa in città alla fine degli anni ottanta, con l’arrivo della musica house e del clubbing contemporaneo. Sempre nel Diamond Dogs in questi anni possiamo incontrare i gruppi napoletani che fanno parte di una nuova corrente estetica locale: la Vesuwave, la new wave del Vesuvio. Erano artisti che guardavano alla musica oltremanica per emancipare il suono partenopeo ormai paludato nelle dinamiche del Neapolitan Power. Tra i gruppi della Vesuwave che si esibiscono al Diamond Dogs troviamo Bisca, Les Bandards Foux, Rhythmotion, i primi Almamegretta, Cibo, Terrapin, Contropotere. Dal resto d’Italia arrivano i Detonazione, i Diaframma, i Liquid eyes, i Not Moving. Passano nel locale diversi gruppi importati da Londra, i Christian Death, Jazzbutcher, Living in Texas, i Playn Jayn e nel novembre dell’84 i ragazzi del Diamond Do",
 "L'Indiependente — Spacca Napoli #3 (punk, 1979–1983)": " Spacca Napoli 3 | L’underground dal punk ai centri sociali Fotografia di Toty Ruggieri: Cristian Death al Diamond Dogs Musica Spacca Napoli 3 | L’underground dal punk ai centri sociali Il viaggio in tre parti tra i suoni e le storie di Napoli si conclude con la Napoli underground: affondiamo nel cuore punk di Napoli, tra i suoi locali e i suoi sotterranei, per arrivare a perderci tra i suoni di band storiche come 99 Posse, Almamegretta e 24 Grana. Qui potete trovare la parte uno, e qui la parte due del viaggio. Buona immersione. Ernesto Razzano è autore di Firenze lo sai, suo primo romanzo uscito per Edizioni 2000diciassette. Scrive di musica per magazine e periodici ed è fondatore del Morgana Music Club di Benevento. Si ringrazia Toty Ruggieri per le fotografie al Diamond Dogs. In copertina il concerto dei Cristian Death. --- Napoli Sotterranea Ma Napoli ha anche una potente anima underground che trova posto nei cunicoli, nei canali e nei passaggi del suo sottosuolo, stratificato nei secoli da storie e civiltà diverse. Da New York prima e da Londra poi, intanto, parte la rivoluzione del punk , che da fenomeno musicale si trasforma anche in subcultura giovanile, facendo presa sui giovanissimi che trovano finalmente un nuovo modo di essere sé stessi senza dover dare troppo conto agli altri. Sono molte le città italiane in cui il punk ha una certa presa: Bologna tra le prime, con Milano, poi Firenze, Pordenone e via via tante altre, ognuna con le sue peculiarità. È l’ultima grande rivoluzione musicale a livello planetario, qualche tempo dopo ci sarà il grunge di Seattle, ma per quanto diffuso non raggiungerà la profondità con cui il punk è stato capace di parlare a una generazione. La filosofia del Do It Yourself, mette in moto il protagonismo, individuale e collettivo, il “fai da te” accorcia i tempi e il rapporto con la possibilità di fare musica, niente più mediazione tramite le grosse etichette, niente attesa di entrare nelle grazie dei critici musicali, né tanto meno una tecnica da perfezionare all’infinito, si comincia ad autoprodurre dischi, a fondare etichette indipendenti e fanzine su cui scrivere e scriversi in modo da stabilire i primi contatti e mettersi “in rete”, tramite la posta, prima ancora del web. Il punk urlava, io ci sono, noi ci siamo, eccoci. E andava oltre la musica perché i seguaci di questo nuovo genere avevano un loro abbigliamento, un proprio codice di riconoscimento, dal taglio di capelli, con la cresta o colorato, a jeans strappati, anfibi, giubbini di pelle, spille da balia appuntate nella carne viva, borchie appese agli indumenti e un dito medio alzato al sistema a definire distanza e territorio. A Napoli sono pochi inizialmente i punk, non hanno ancora luoghi ben definititi dove trovarsi per suonare o per ascoltare musica. Sono pochi si, ma visibili. Così i primi ritrovi sono i muretti di Piazza del Gesù, Piazza dei Martiri e De Marco, un negozio di dischi, al Vomero, quartiere in cui si trova anche un altro ritrovo, che seppur piccolo è molto frequentato e si chiama Cantina Sica. “A Napoli e provincia i dissidenti al culto delle Timberland, del Moncler e dei fratelli Vanzina, si compone di varie tribù. Ci sono i punk di Mugnano, quelli di Piazza del Gesù, quelli di Cavalleggeri, che sono un po’ più fashion alla Clash, i metallari della Napoli bene che vanno ogni estate a fare delle presunte vacanze studio a Londra, quelli che invece vengono dalle periferie, i dark del Vomero, e poi ci siamo noi fricchettoni, i figli dei fiori fuori tempo massimo, gli ultimi giapponesi di Woodstock.” Sacha Ricci (99 Posse) in Curre Curre Guagliò di Rosario Dello Iacovo diamond dogs 1982/1984 Foto di Tony Ruggieri: Bisca al Diamond Dogs Quella che viene riconosciuta come la prima punk band a esibirsi in pubblico va sotto il nome di Underage . Avrà una vita breve, solo qualche anno, ma per più di un motivo sarà una vera apripista, con la manciata di canzoni dell’Ep dal titolo “Africani, Marocchini, Terroni”, stampato dall’etichetta emiliana Attack Punk Records, artefice anche delle uscite dei CCCP – Fedeli alla Linea di Giovanni Lindo Ferretti e Massimo Zamboni. “Il primo vero concerto punk a Napoli: 13 aprile 1982. La venue prescelta è lo ZX di Via Atri e il pubblico di indemoniati arriva perlopiù dalla fauna di De Marco Dischi e delle fanzine come Megawave” Caludio Biazzetti A creare e curare la fanzine Megawave (ma anche Shock e Hate Again tra le prime del sud), è Davide Morgera, batterista degli Underage, (e ancor prima degli Elettroencefalogramma) uno dei primi punk napoletani. Ma se i primi gruppi cominciano a formarsi, le registrazioni cominciano a girare e i posti di ritrovo ad essere sempre più frequentati, mancano ancora dei luoghi appositamente dedicati. Non sono ancora molti i punk a Napoli, ma di sicuro cominciano a farsi notare per le strade, attirano sempre più l’occhio di chi in qualche modo intravede una propria possibilità di essere fuori da schemi e panni che cominciano a essere stretti. I piccoli gruppi nei vari quartieri aumentano e cominciano a riconoscersi. “Siamo in coda agli anni di piombo. Sono gli anni dell’epidemia di eroina che, come l’epidemia di crack negli Stati Uniti, cancellerà un’intera generazione di attivisti politici; dello stato che scende a patti con la camorra tramite i servizi segreti, mentre la realtà di tutti i giorni è diventata un’esperienza kafkiana. Non è affatto un eccesso di immaginazione affermare che il tentativo di rivoluzionare il caos che soffocava la città da parte dei giovani di Napoli, come i frequentatori del Diamond Dogs, abbia gettato le basi di fenomeni a venire.” Paolo Pontoniere in Diamond Dogs, Officina post industriale -1984-1987, Napoli di Toty Ruggieri diamond dogs 1982/1984 diamond dogs 1982/1984 Foto di Tony Ruggieri: Diamond Dogs Creste punk, disoccupazione e creatività “Un portone di ferro arrugginito a Cavone San Gennaro, Rione Sanità, apriva un accesso a un cordone ombelicale di scale ripide che, riemergendo dalle viscere della colina, collegava il club al resto della città sovrastante e questa al suo ventre stratificato: agli acquedotti abbandonati d’epoche immemori, alle chiese paleocristiane, alle cave di tufo Greco-romane, ai cunicoli militari dei Borboni, alle stalle e ai postriboli della Corte dei Miracoli. In fondo si andava a sbattere nella cabina di regia: l’abitacolo di un camion nel quale ogni notte si accavalcavano, registi e registrati, a creare visoni paleo-tecnologiche di futuri digitali che, inconsapevoli della loro valenza, anticipavano di almeno un ventennio l’avvento di un nuovo universo mediatico. La caverna maggiore tutta graffitata in stile mittel-europeo ospitava concerti, danze e azioni teatrali.” Mattia Ruffolo [... middle omitted — see footer ...] Ernesto Razzano Share This Previous Article Il ritorno dei Kings of Convenience e i 10 singoli del mese Next Article Playlist | SpaccaNapoli: suoni e storie della città Questo sito usa i cookies per migliorare la tua esperienza. Se vuoi saperne di più, clicca su Policy. Accetto Policy Privacy & Cookies Policy Privacy Overview This website uses cookies to improve your experience while you navigate through the website. Out of these, the cookies that are categorized as necessary are stored on your browser as they are essential for the working of basic functionalities of the website. We also use third-party cookies that help us analyze and understand how you use this website. These cookies will be stored in your browser only with your consent. You also have the option to opt-out of these cookies. But opting out of some of these cookies may affect your browsing experience. NecessarySempre abilitato Necessary cookies are absolutely essential for the website to function properly. This category only includes cookies that ensures basic functionalities and security features of the website. These cookies do not store any personal information. Non-necessary Any cookies that may not be particularly necessary for the website to function and is used specifically to collect user personal data via analytics, ads, other embedded contents are termed as non-necessary cookies. It is mandatory to procure user consent prior to running these cookies on your website. ACCETTA E SALVA Caricamento commenti... ──────── [TRUNCATED] ──────── Showing 8,573 chars (head) + 2,091 chars (tail) of 37,238 total clean characters. Full text saved to: /home/agent/.hermes/cache/web/www.lindiependente.it-99b3ad7468.md To read the omitted middle: read file path=\"/home/agent/.hermes/cache/web/www.lindiependente.it-99b3ad7468.md\" offset=59 limit=200 (the file is the complete page; raise/lower offset to page through it). ─────────────────────────────"
}
print(f"{len(DOCS)} documenti, {sum(len(v) for v in DOCS.values())} caratteri totali")


In [ ]:
import json, os, re, time, urllib.request

PROMPT = '''Sei un estrattore di dati storici. Dal TESTO estrai SOLO i luoghi musicali di Napoli nominati
esplicitamente (negozi di dischi, club, discoteche, teatri, centri sociali, piazze o muretti usati come ritrovo).
Per ognuno restituisci: nome (come nel testo), tipo (negozio_dischi|club|discoteca|teatro|centro_sociale|ritrovo|altro),
via (via o piazza se nominata, altrimenti ""), quartiere (se nominato, altrimenti ""), anni (date o anni nominati,
altrimenti ""), citazione (la frase esatta del testo, massimo 30 parole).
Non inventare nulla. Rispondi SOLO con un array JSON.

TESTO:
---
{testo}
---'''


def chiavi():
    """Vraca prvi dostupni ključ: Colab Secrets pa environment."""
    out = {}
    try:
        from google.colab import userdata
        for k in ("GEMINI_API_KEY", "DEEPSEEK_API_KEY", "GEMMA4_CF_CLIENT_ID", "GEMMA4_CF_CLIENT_SECRET"):
            try:
                v = userdata.get(k)
                if v:
                    out[k] = v
            except Exception:
                pass
    except Exception:
        pass
    for k in ("GEMINI_API_KEY", "DEEPSEEK_API_KEY", "GEMMA4_CF_CLIENT_ID", "GEMMA4_CF_CLIENT_SECRET"):
        if k not in out and os.environ.get(k):
            out[k] = os.environ[k]
    return out


def _gemini(prompt, k):
    url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent"
    req = urllib.request.Request(url, data=json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode(),
                                 headers={"Content-Type": "application/json",
                                          "x-goog-api-key": k["GEMINI_API_KEY"]}, method="POST")
    with urllib.request.urlopen(req, timeout=240) as r:
        d = json.loads(r.read().decode())
    return d["candidates"][0]["content"]["parts"][0]["text"]


def _deepseek(prompt, k):
    req = urllib.request.Request("https://api.deepseek.com/chat/completions",
                                 data=json.dumps({"model": "deepseek-chat", "temperature": 0.1,
                                                  "max_tokens": 3000,
                                                  "messages": [{"role": "user", "content": prompt}]}).encode(),
                                 headers={"Content-Type": "application/json",
                                          "Authorization": "Bearer " + k["DEEPSEEK_API_KEY"]}, method="POST")
    with urllib.request.urlopen(req, timeout=240) as r:
        d = json.loads(r.read().decode())
    return d["choices"][0]["message"]["content"]


def _gemma(prompt, k):
    req = urllib.request.Request("https://gemma4.syntagent.com/v1/chat/completions",
                                 data=json.dumps({"model": "gemma-4-26b-a4b-nvfp4", "temperature": 0.1,
                                                  "max_tokens": 2500,
                                                  "messages": [{"role": "user", "content": prompt}]}).encode(),
                                 headers={"Content-Type": "application/json",
                                          "CF-Access-Client-Id": k["GEMMA4_CF_CLIENT_ID"],
                                          "CF-Access-Client-Secret": k.get("GEMMA4_CF_CLIENT_SECRET", "")},
                                 method="POST")
    with urllib.request.urlopen(req, timeout=300) as r:
        d = json.loads(r.read().decode())
    return d["choices"][0]["message"]["content"]


def chiama(prompt, k):
    """Prova i backend in ordine e passa al successivo se uno è bloccato (quota, 429, rete)."""
    tentativi = [("GEMINI_API_KEY", _gemini), ("DEEPSEEK_API_KEY", _deepseek),
                 ("GEMMA4_CF_CLIENT_ID", _gemma)]
    errori = []
    for nome, fn in tentativi:
        if nome not in k:
            continue
        for prova in range(2):
            try:
                return fn(prompt, k)
            except Exception as e:
                errori.append(f"{nome}:{type(e).__name__}")
                time.sleep(2 * (prova + 1))
    raise RuntimeError("nessun backend ha risposto — " + ", ".join(errori))


def estrai(testo, k):
    raw = chiama(PROMPT.replace("{testo}", testo), k)
    m = re.search(r"\[.*\]", raw, re.S)
    if not m:
        return []
    try:
        data = json.loads(m.group(0))
    except Exception:
        return []
    return [r for r in data if isinstance(r, dict) and len(str(r.get("nome", "")).strip()) > 2]


K = chiavi()
print("chiavi trovate:", ", ".join(K) if K else "nessuna")


In [ ]:
import pandas as pd

if not K:
    print("Nessuna chiave: aggiungi GEMINI_API_KEY (o DEEPSEEK_API_KEY) nei Colab Secrets e riesegui. "
          "Sotto resta il risultato dell'ultima esecuzione, prodotto dallo stesso codice.")
    righe = []
else:
    righe = []
    for nome_doc, testo in DOCS.items():
        for i in range(0, len(testo), 4200):  # blocchi sovrapposti
            try:
                trovate = estrai(testo[i:i + 5400], K)
            except Exception as e:
                print("  errore:", nome_doc[:40], type(e).__name__, str(e)[:80])
                continue
            for r in trovate:
                r["documento"] = nome_doc
                righe.append(r)
            print(f"  {nome_doc[:38]:38} blocco {i//4200}: +{len(trovate)}")
            time.sleep(1)

print("estrazioni grezze:", len(righe))


In [ ]:
# PASS_REGEX — secondo passaggio deterministico: prende i nomi costruiti come "lo ZX di via Atri",
# "si chiama Cantina Sica", "aprono il Diamond Dogs". Serve a non perdere i casi che il modello salta.
PASS_REGEX = [
    (r"(?:il|lo|la)\s+([A-Z][\w'’]{1,22})\s+di\s+((?:Via|Viale|Piazza|Piazzetta)\s+[A-Z][\w'’ ]{2,24})", "club"),
    (r"(?:si chiama|si chiamava)\s+([A-Z][\w'’ ]{2,24})", "ritrovo"),
    (r"(?:aprono|apre|inaugurano)\s+(?:il|la|lo)\s+([A-Z][\w'’ ]{2,24})", "club"),
    (r"(?:il|lo|la)\s+([A-Z][\w'’ ]{1,22})\s*,?\s+(?:aperta|aperto|chiusa|chiuso)", "discoteca"),
]
CUE_TIPO = [("venue", "club"), ("concerto punk", "club"), ("concerto", "club"),
            ("negozio di dischi", "negozio_dischi"), ("discoteca", "discoteca"), ("teatro", "teatro"),
            ("centro sociale", "centro_sociale"), ("muretto", "ritrovo"), ("ritrovo", "ritrovo"),
            ("club", "club"), ("locale", "club")]

def frasi(testo):
    testo = re.sub(r"\s+", " ", testo)
    return [s.strip() for s in re.split(r"(?<=[\.\!\?])\s+", testo) if 25 < len(s) < 600]

aggiunte = []
for nome_doc, testo in DOCS.items():
    for s in frasi(testo):
        basso = s.lower()
        tipo = next((t for cue, t in CUE_TIPO if cue in basso), None)
        if not tipo:
            continue
        anno = re.search(r"\b(19[5-9]\d)\b", s)
        for pattern, tipo_p in PASS_REGEX:
            for m in re.finditer(pattern, s):
                nome = m.group(1).strip()
                via = m.group(2).strip() if m.lastindex and m.lastindex >= 2 else ""
                aggiunte.append({"nome": nome, "tipo": tipo_p, "via": via, "quartiere": "",
                                 "anni": anno.group(1) if anno else "",
                                 "citazione": s[:300], "documento": nome_doc, "metodo": "regex"})
for riga in righe:
    riga.setdefault("metodo", "llm")
print("aggiunte dal passaggio regex:", len(aggiunte), "|", [a["nome"] for a in aggiunte][:8])
righe = righe + aggiunte


In [ ]:
def chiave(n):
    return re.sub(r"\W+", "", str(n).lower())


unici = {}
for r in righe:
    kk = chiave(r.get("nome"))
    if kk.startswith("demarco"):
        kk = "demarco"
    if kk not in unici:
        unici[kk] = {"nome": str(r.get("nome", "")).strip(), "tipo": r.get("tipo") or "altro",
                     "via": r.get("via") or "", "quartiere": r.get("quartiere") or "",
                     "anni": r.get("anni") or "",
                     "citazione": (r.get("citazione") or r.get("citatio") or "")[:300],
                     "documento": r.get("documento", ""), "metodo": r.get("metodo", "llm")}
    else:
        for campo in ("via", "quartiere", "anni"):
            if not unici[kk][campo] and r.get(campo):
                unici[kk][campo] = r.get(campo)

estratte = pd.DataFrame(list(unici.values()))
if len(estratte):
    display(estratte[["nome", "tipo", "via", "quartiere", "anni", "metodo", "citazione"]])
    print("luoghi unici estratti:", len(estratte))
else:
    print("niente da mostrare — manca la chiave o il testo")


### The real table: 24 places, 15 located

The corpus below is not two pages but eight: Rolling Stone/Capasso, *Spacca Napoli #3*, the Madre and Frizzifrizzi
pieces on the Diamond Dogs, Capobianco's memoir, the Radio Kiss Kiss oral-history transcript, the Wikipedia entry
and the TTC Club history. The extractor ran on all of them, and the result — deduplicated by name — is 24 places,
of which 15 can be located; the other nine have a name and a year but no address, which is exactly the work the
Pagine Gialle and the Emeroteca would settle.

Every row carries the source, the quote and an evidence level: **B** when two independent documents agree,
**C** when only one does. Nothing here comes from a model alone.


In [ ]:
import base64, gzip, io
import pandas as pd

# la tabella prodotta dall'estrattore sul corpus (CSV compresso, incorporato nel notebook)
REALI_B64 = "H4sIAAAAAAAC/+1ZzW7jyBG+G/A7dBQgMwPQ+pco+aaVkMlkx4Yx9k6AXIwy2ZY6Jrs5ZFNeGznsQ+SSW455hZznTfZJ8lU3RVGyBzuj3cnuIQasH5JdXfVV1VdVLW1SGViVmWCtKPhQUm6VzGVAWqsgIRskRgdZLiNVKKNlkKi1TBITRMrSo7sSm6hMpbYqWEpzfWu0lUGZJ8dHC0Wp0bFYmGURREl5E8xpjRXikrR4LbWm3AT4rOzHfwW96WR4gpcwGHbbk3F/2gt6w3Z/OJlMnGbfBK015RFZI1QiMpNblhQrcSvz3AjK83K5VJCFz4l4ZiOhZSKi0uRYhU+5qh5wu4tCiSKSOpZrarfb/GxiRGFNriIjEhNRIkWJ2/kyNyXb1LCtFbTmlFFRYJMrqcVcAcJC/PjDP4QsbE4WOr0zSaL0Ulw6tf++s16ciDOKodc5ZSZRe3dPRUKbO00NYrnEFXaTeDHpOo2luDXYC6BcGfsgIOjtizc6VjJjy+AXp9NlRlFUi/z9AOpnK6zTpzVs7Q1q2EWJC7OWuQqqFYE4K7WKVEYJQyewfmVtVpx2Ovf39+2ULdHuybayHQZOxwS0O7E36iSGUcdH7yhWRnyrisK9BDl/D94r8k5TFJnUbw9ssTsCJBxzgIxdgIzCoQ+Q/qBfxcclFVAHT5sbRRpuY4fzKhHlkhhFv2e0kgIBDUQ4kjTiX8SU4HasishYCWxqtYQULsrWSuJ9oXIJZ8JTM2RJYREbB7i+oeGJ2Ff6VFxBvYWJsM3v8PBZWSgo9AdxJlMEI4kO9sHOQBLPP9Kp2ANyG0T7N07EX9SdyiT80HD5pxEX71yKzBIL42YADBFGO97f9XxByZoYnai2hiPAruTJPT2c3OMNxHJ8tPV5DfhP+H2w9fu42x1Wfh9tHN86944e1I6CXxequBN/NtGdfOB8h3shLSWR0vdq62gXqNKSJl6z1cylE4mi3CwrJIeL/jRcra8dCb9Vd16SLfOYHl4UYs4c74ie1bhAKZHaZJJ9OJ469/WnY0/rw8nAeW8eAPc9ESBjZnkPdi1F1JTl8xoSPx+/XYC2mokL9fhobimJuJI55P4o81hp0rHZZ7pecCBC0Url6oZ38ui4oO7hDea36sgEyRJiJQPXSmQ2gk+JTCFwlPj4bxQ1FKSGJPHSElZoEBIzF0qDd3aqUBbAFOJPiPBkG+AIACD9zjxQ8mrDjL3Wl0B4kO1/lVkmEfTbsDgDbYG8YDwHRC8chC4iur3ptIqI1gXSiMSHUqF2ecNrMXVNPlfLlWXDGzJbB0fELJHYUseoeBv9cB0Jos3awXthCnzlNqn58ezAgFjRncwb6eL/5kHT5MZDIkYFduQGEkKVL0r8I1pQ0BG3fEUWHBVIcCW4QXORsMyJhSGeCqMJzZHrezTj9tXd7snqjPOsquwc85MQRD7telsXxDE4CYV7n3Y5yJ8psAx+U9r/SHMZN/SedlnvvtfbdQteY//eb6goY/GSUueKwiCvzZLt4QcWCgn6tuTkfPXVjbgyelnKKraqts2l22AUhlVfPRg3un1Hw34VK1stOSSbNk3inNIMrIMK88aCP6n1M7yBjhWAiu+0aTglHFcEOtPc0e1VB3R9mtDRr5HBnCBMsVK41ciQGwjfdE4N4dWVStAcHXNeX2y2hJTYXLZbwaHN2a5TUydFpl5GG2W7Y5tSrp3W13dY717aK5tukNlV9f/g7IAzw8SDKUa8JxSNJjjTCpw3GmUT5VdFkUEZ9gAwSaZ43yaHe0imylo3Q6EGU0p2M07s7OKGC8ySJorKjNautGvXQaRuLXLLW9ToPH9lrK6u5o1KtHbtEWZ9N+S/x3AEL/shM+zCuxg1mUimw7A7ZSIZdMNwtKnbyk/KqsLQ0TfEBxjXoYYLPjQ0mns7i6md/Kweoa7hwcGU0dnZn+96FdDhviVf+8lJr7X2MxwkFQa8zMQr1rxEFnAciQhOYEmbabm13wluTOWmEFq8R/cnmb0UJ0fwDJ1Nut3+cMtmSz8oM4/FwMm0700eY7wEvOyAfrc/6PS6nX7YsTZiiDvHRxeKHh+9Ha9l8fE/Qa6QTGsTPLnhWXs4Cn3bPOoP+1uwOW2V8GsV7GdgEV5cQpXrWfalba8oV03RS8KlC8lfImD8OacFW/z2xYvzkm2oxo+y4JZtv4Xu14i2ggYOtTpPkdje2lSw8chXsO5o9NvDYqvuZ48WjMRm30DLpUEMX/OIuoLNVQb6MOhP+s703qg72CnerXo9uEpUMtjsSkwzkb7MsGp/8V3BB37110sQHK2fTEjT3YRhy+akLWwWl6Cm2ru7Zg0GfqTvjbvhrlm12sGWjnkwjDAagWVZGCGpmXRhN1ceIzZ7eCpGEJZ5xfCGW8zUYAAWt7n8UMJ8piBH2J7URVPbXwEq5iq5XMpGIgQ7V/3MNAwd9/YA3WgXMBiVJC7gd5Z5LDgpAFNmfvzhn4AEGXtLxQpDg3DHX/ME377U6Nn5GXrDhQTD89De3FXMwPogzG9oqZ85N+x1HQdc5Soxy4eqV61AU9oR+eZwswqTyaAKk12buXP1QjAYsSGRO2bxye4FipdugnLkQKk7W0BWFoaPTF8rTEYLQDLLmd75cBlcguz5GwcUJmyyam2qMyQ+I6ZXX3zW0/oFo+SiTArKt9V6zihz1BrfETyHXTiaOuxGjWoNQ72oTdndE/QzTHwqzBOkcSdO58a9PldYh0OfB7e3yFSPeILmqwqOOXIUDz5n4GDombHPPxfsjTVPxLlGZMsnnCxe8uEmVwKeHBg9a2avH26nouOjS7nU5JN2FkV8jJ9Xv6xErsm+LkykKJH1KUFrc9bji557imseP4UeBwO/dWcIiGnkJWhXfHqLqmND8fJHbhmnCDpWqk5AEaRsiVvILdsBoc9JLompeVYW3IoG1n2FLRpZmd74X2RY3cmwshClnJaIGN9K7vxYwj/9cA9Z09G5iu6YeGCM0u4crMikQ4yTHFpDJbTope827Y4qh5nzXwRcwWo0GwAA"

reali = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(REALI_B64))))
reali = reali.fillna("")
print(f"{{len(reali)}} luoghi estratti · {{int((reali.lat != '').sum())}} localizzati · "
      f"livello B: {{int((reali.livello == 'B').sum())}} · C: {{int((reali.livello == 'C').sum())}}")
display(reali[["nome", "tipo", "via", "quartiere", "anni", "livello", "documenti"]].rename(
    columns={{"documenti": "fonti"}}))


In [ ]:
GAZ = {
    "via atri": (40.8515, 14.2545), "via costantinopoli": (40.8516, 14.2517),
    "cavone san gennaro": (40.8576, 14.2520), "sanità": (40.8576, 14.2520),
    "vomero": (40.8439, 14.2255), "chiaia": (40.8330, 14.2280),
    "piazza del gesù": (40.8479, 14.2553), "piazza dei martiri": (40.8345, 14.2415),
    "acerra": (40.9461, 14.3750),
}


def coord_riga(r, i):
    for campo in ("via", "quartiere", "nome"):
        t = str(r.get(campo) or "").lower()
        for k, c in GAZ.items():
            if k in t:
                rnd = random.Random(i)
                return [c[0] + rnd.uniform(-0.002, 0.002), c[1] + rnd.uniform(-0.002, 0.002)]
    return None


if len(estratte):
    import folium
    m2 = folium.Map(location=[40.8518, 14.2580], zoom_start=12, tiles="OpenStreetMap",
                    attr="© OpenStreetMap contributors")
    locati = 0
    for i, r in estratte.iterrows():
        c = coord_riga(r, i)
        if not c:
            continue
        locati += 1
        folium.Marker(
            location=c,
            popup=folium.Popup(f"<b>{r['nome']}</b><br>{r['tipo']} · {r['via'] or r['quartiere']}<br>"
                                f"<i>{r['citazione'][:200]}</i><br><small>{r['documento']}</small>", max_width=330),
            tooltip=str(r["nome"])).add_to(m2)
    m2.save("estrazioni_reali_map.html")
    print(f"localizzati {locati} di {len(estratte)} — mappa scritta in estrazioni_reali_map.html")
    display(m2)
    estratte.to_csv("estrazioni_reali.csv", index=False, encoding="utf-8")
    print("tabella salvata: estrazioni_reali.csv")
else:
    print("nessuna estrazione da mappare")


### The map of the real data

The ten places above, geolocated and published — the same file the cell writes as `estrazioni_reali_map.html`.
Grey dots would be models; these are not models. Every marker carries the sentence it was extracted from, so a
click is also a citation.
 

In [ ]:
REALI_MAP_URL = "https://bperak.github.io/napoli-1980-map/reali.html"
try:
    from IPython.display import HTML, display
    display(HTML(f'<div style="font:12px sans-serif;color:#555;margin:4px 0 6px">'
                  f'live map of extracted places: <a href="{REALI_MAP_URL}" target="_blank">{REALI_MAP_URL}</a></div>'
                  f'<iframe src="{REALI_MAP_URL}" width="100%" height="560" '
                  f'style="border:1px solid #ccd;border-radius:6px" loading="lazy"></iframe>'))
except Exception:
    print("Apri la mappa dei dati estratti:", REALI_MAP_URL)


## 11. Where the sources are, and how to reach them

Twenty-six sources, each with the practical path: what it holds, where it is, what it costs, how long it takes and how many places it can yield. The full table is a Google Sheet, the prose version is a Google Doc, and both are public — anyone with the link can read them without an account:

- Sheet: <https://docs.google.com/spreadsheets/d/1jQ5VB9txOYP1GgMXmGyU-HreoMl1_PpHkB_c_U31Y0U/edit>
- Repository (code, data, maps): <https://github.com/bperak/napoli-1980-musical-memory>
- Guida (Doc): <https://docs.google.com/document/d/1TtKjNwp9UV_uakyR7oYdjGHGNtNiaIuowXkZwF8XiVE/edit>

| # | Fonte | Come si arriva | Costo e tempo | Resa | Difficoltà |
|---|---|---|---|---|---|
| 1 | Rolling Stone Italia — «Ten Cities: la storia del clubbing a Napoli» (D. Capasso) | Online, aperto: rollingstone.it/musica/ten-cities-la-storia-del-clubbing-a-napoli/543958 — leggere le parti 1975-1990 | gratis, 20 min | 8-10 luoghi | 1/5 |
| 2 | L'Indiependente — «Spacca Napoli #3» (punk 1979-1983) | Online, aperto: lindiependente.it/spacca-napoli-3-underground | gratis, 15 min | 6-8 luoghi | 1/5 |
| 3 | Madre Museum — scheda evento «Diamond Dogs» | Online: madrenapoli.it/calendario/diamond-dogs | gratis, 10 min | 1-2 luoghi | 1/5 |
| 4 | Frizzifrizzi — «Diamond Dogs: la Napoli underground anni '80» | Online: frizzifrizzi.it/2016/05/26/diamond-dogs | gratis, 15 min | 2-3 luoghi | 1/5 |
| 5 | Salvatore Capobianco «The Doctor» — memoir del dj | Online: salvatorecapobianco.it/the-way-we-were | gratis, 30 min | 10-15 luoghi | 1/5 |
| 6 | Musica e Memoria — testimonianze orali (Radio Kiss Kiss) | Online: musicaememoria.com/testimonianza_radio_kiss_kiss.htm | gratis, 20 min | 3-5 luoghi | 1/5 |
| 7 | Wikipedia (it) — voci su radio e locali napoletani | Online: it.wikipedia.org (verificare sempre le note in fondo) | gratis, 10 min | 2-4 luoghi | 1/5 |
| 8 | Disco & Dancing — archivio dei locali (Campania) | Online: discodancing.it/locali/campania | gratis, 10 min | 5-8 nomi | 2/5 |
| 9 | Dusty Dancing — discoteche abbandonate in Campania | Online: dustydancing.com/category/discoteche-abbandonate-campania | gratis, 20 min | 3-5 luoghi | 1/5 |
| 10 | Discogs — etichette e stampe napoletane | Online: discogs.com → Label → ricerca «Napoli»; filtro anni 1975-1990 | gratis, 30 min | 3-6 nomi | 2/5 |
| 11 | Internet Archive — testi che nominano Napoli 1975-1992 | Online: archive.org → ricerca avanzata, «Napoli» + intervallo date; 209 testi già individuati | gratis, 45 min | 5-10 luoghi | 2/5 |
| 12 | Google Books — guide e annuari | Online: books.google.com; l'API va a singhiozzo (HTTP 429) | gratis, 30 min | 3-6 luoghi | 2/5 |
| 13 | Archivio RAI / YouTube — filmati di concerti | Online: ricerca «Napoli concerto 1982»; verificare titolo e data di caricamento | gratis, 40 min | 1-3 luoghi | 3/5 |
| 14 | Gruppi Facebook (Vomero anni '80, Retrodance, Napoli Underground) | Richiede account e consenso dell'amministratore del gruppo; mai citare senza permesso | gratis, 60 min | 5-10 nomi | 4/5 |
| 15 | OpenStreetMap + Photon/Nominatim | API pubbliche: photon.komoot.io (usata qui) e nominatim.openstreetmap.org (limite 1 req/s) | gratis, automatico | geocodifica | 1/5 |
| 16 | Biblioteca Nazionale «Vittorio Emanuele III» — sezione napoletana | Palazzo Reale, Piazza Plebiscito 1, Napoli; tessera gratuita; chiedere in sala «annuari telefonici, collocazione Napoli» | gratis, 2-3 ore | 15-25 luoghi | 3/5 |
| 17 | Emeroteca Tucci | Via L. Giordano (c/o Università Suor Orsola Benincasa), Napoli; prenotare il posto; fotocopie a pagamento | 2-4 €/pagina, 3-4 ore | 10-20 luoghi | 3/5 |
| 18 | Archivio di Stato di Napoli | Piazzetta Grande Archivio 5, Napoli; registrazione annuale; inventari in sala studio | gratis, 3-5 ore | 5-15 luoghi | 4/5 |
| 19 | Comune di Napoli — SUAP / ufficio commercio | Palazzo San Giacomo, Piazza Municipio; istanza di accesso agli atti (L. 241/1990) con oggetto preciso | gratis, 30-60 giorni | 10-20 luoghi | 5/5 |
| 20 | SIAE — sede di Napoli | Viale Gramsci (Chiaia), Napoli; richiesta scritta motivata per ricerca storica | gratis, 2-4 settimane | 10-20 luoghi | 4/5 |
| 21 | Camera di Commercio di Napoli — Registro Imprese | Online (registroimprese.it, visura a pagamento) oppure sportello in Piazza Bovio | 1-3 €/visura, 20 min | 10-20 luoghi | 2/5 |
| 22 | Archivio fotografico Carbone / Fondazione Morra / Madre | Contattare l'archivio via mail; consultazione su appuntamento; diritti d'uso separati | gratis-50 €/immagine | 5-10 luoghi | 3/5 |
| 23 | Archivio Toty Ruggieri (fotografo del Diamond Dogs) | Contatto tramite Madre/Museo o il libro «Diamond Dogs» (2016) | da concordare | 2-5 luoghi | 3/5 |
| 24 | Fanzine — «Megawave» (D. Morgera) e raccolte private | Collezioni private e biblioteche specializzate; chiedere alla Biblioteca Nazionale la collocazione dei fondi minori | gratis, 2-3 ore | 5-10 luoghi | 4/5 |
| 25 | Archivi radiofonici — Radio Kiss Kiss, RAI Campania | Contattare l'ufficio stampa dell'emittente; per la RAI, mediateca regionale | da concordare | 2-5 luoghi | 4/5 |
| 26 | Storia orale — dj, titolari, famiglie | Contatti diretti (Capobianco, Gigi D'Aria, famiglia Niespolo per il Kiss Kiss); registrare con consenso scritto | gratis, 1-2 ore per intervista | 5-10 luoghi | 2/5 |

The three evidence levels used throughout: **A** a period document with an address (Pagine Gialle, a licence, a company record); **B** two independent sources that agree; **C** one source, or a source without an address; **D** the model alone, which stays a hypothesis.
